# 03 Data Preparation — availability-aware alignment

สร้างตารางพยากรณ์ ณ **วันที่ 15 ของเดือน t+2** ใช้ `h=3` เป็นผลหลัก `h=2` เป็น diagnostic และ `h=4–6` ใช้ตอบ RQ4(ก) ว่าเตือนได้ไกลแค่ไหน การแก้ไขนี้แทน notebook รุ่นเดิมที่สมมติ forecast origin วันที่ 25 ของ t+1

Archive ข่าว สนค. ใช้ยืนยันได้เฉพาะช่วงที่มีรายการ วันที่ 15 ของ t+2 จึงเป็น conservative proxy และต้องรายงานเป็นข้อจำกัด ค่าเฉลี่ยเต็มเดือน t+1 เป็น recent-information feature ส่วน cutoff วันที่ 24 ใช้ทำ sensitivity check

In [1]:
from __future__ import annotations
import hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'PROJECT_SCOPE.md').exists())
RAW_DIR, INTERIM_DIR = ROOT / 'data/raw', ROOT / 'data/interim'
PROCESSED_DIR, METRICS_DIR = ROOT / 'data/processed', ROOT / 'logs/metrics'
for directory in (INTERIM_DIR, PROCESSED_DIR, METRICS_DIR): directory.mkdir(parents=True, exist_ok=True)
GROUPS = {'100': 'สินค้าเชื้อเพลิง', '318': 'ปุ๋ยฯ', '402': 'นมและผลิตภัณฑ์นม'}
HORIZONS, PRIMARY_HORIZON, FORECAST_DAY = (2, 3, 4, 5, 6), 3, 15  # h=2 diagnostic, h=3 หลัก, h=4-6 สำหรับ RQ4(ก)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): digest.update(chunk)
    return digest.hexdigest()
sources = json.loads((RAW_DIR / 'sources.json').read_text(encoding='utf-8'))['sources']
def latest(prefix): return next(x for x in reversed(sources) if x['source_id'].startswith(prefix))
src = {'IMI': latest('imi-monthly-'), 'DEXTHUS': latest('dexthus-'),
       'DCOILBRENTEU': latest('dcoilbrenteu-'), 'TPSO_RELEASE': latest('tpso-release-calendar-')}
for item in src.values(): assert sha256(RAW_DIR / item['file']) == item['sha256']
display(pd.DataFrame([{'source': k, 'file': v['file'], 'sha256_ok': True} for k, v in src.items()]))

# 1) หลักฐานวันเผยแพร่
release = pd.read_csv(RAW_DIR / src['TPSO_RELEASE']['file'], parse_dates=['reference_month', 'release_datetime'])
release['release_date'] = release.release_datetime.dt.normalize()
release['cutoff_day15_t2'] = release.reference_month + pd.offsets.MonthBegin(2) + pd.Timedelta(days=14)
release['buffer_days'] = (release.cutoff_day15_t2 - release.release_date).dt.days
release['released_in_t2'] = release.release_date.dt.to_period('M') == (release.reference_month + pd.offsets.MonthBegin(2)).dt.to_period('M')
release_summary = pd.DataFrame([{'records': len(release),
    'reference_start': release.reference_month.min().date(), 'reference_end': release.reference_month.max().date(),
    'lag_min_days': int(release.lag_days_after_month_end.min()),
    'lag_median_days': float(release.lag_days_after_month_end.median()),
    'lag_max_days': int(release.lag_days_after_month_end.max()),
    'latest_release_day_when_in_t2': int(release.loc[release.released_in_t2, 'release_date'].dt.day.max()),
    'minimum_buffer_to_day15': int(release.buffer_days.min())}])
assert len(release) == 21 and (release.release_date <= release.cutoff_day15_t2).all()
display(release_summary)

# 2) IMI และข้อมูลภายนอกรายเดือน
raw = pd.read_csv(RAW_DIR / src['IMI']['file'], dtype={'commodityCode': 'string'})
raw['commodityCode'] = raw.commodityCode.str.zfill(3)
raw['month_ce'] = pd.to_datetime(dict(year=raw.year - 543, month=raw.month, day=1))
imi = (raw[raw.commodityCode.isin(GROUPS)][['commodityCode','commodityNameTH','month_ce','index']]
       .rename(columns={'commodityCode':'commodity_code','commodityNameTH':'commodity_name_th','index':'imi_index'})
       .sort_values(['commodity_code','month_ce']).reset_index(drop=True))
assert len(imi) == 960 and not imi.duplicated(['commodity_code','month_ce']).any() and (imi.imi_index > 0).all()
external_parts = []
for series_id in ('DEXTHUS', 'DCOILBRENTEU'):
    daily = pd.read_csv(RAW_DIR / src[series_id]['file'], na_values=['.'])
    daily['observation_date'] = pd.to_datetime(daily.observation_date)
    daily = daily[(daily.observation_date >= '2000-01-01') & daily[series_id].notna()].copy()
    daily['month_ce'] = daily.observation_date.dt.to_period('M').dt.to_timestamp()
    full = daily.groupby('month_ce').agg(full_mean=(series_id,'mean'), full_n=(series_id,'size'), full_max_date=('observation_date','max'))
    d24 = daily[daily.observation_date.dt.day <= 24].groupby('month_ce').agg(
        cutoff24_mean=(series_id,'mean'), cutoff24_n=(series_id,'size'), cutoff24_max_date=('observation_date','max'))
    monthly = full.join(d24).reset_index(); monthly.insert(0, 'series', series_id); external_parts.append(monthly)
external = pd.concat(external_parts, ignore_index=True)
external.to_csv(INTERIM_DIR / 'external_monthly_aggregates.csv', index=False)
display(external.groupby('series').agg(months=('month_ce','size'), start=('month_ce','min'), end=('month_ce','max'),
                                            min_full_n=('full_n','min'), min_cutoff24_n=('cutoff24_n','min')))

# 3) Features ถึง t และ recent-information ของ t+1
parts = []
for _, group in imi.groupby('commodity_code', sort=False):
    group = group.copy(); log_i = np.log(group.imi_index)
    for w in (1,2,3,6,12): group[f'imi_log_change_{w}m_pct'] = 100 * (log_i - log_i.shift(w))
    group['imi_volatility_6m_pct'] = (100 * log_i.diff()).rolling(6, min_periods=6).std(ddof=1)
    parts.append(group)
base = pd.concat(parts, ignore_index=True)
for series_id, prefix in (('DEXTHUS','fx'), ('DCOILBRENTEU','brent')):
    m = external[external.series == series_id].sort_values('month_ce').set_index('month_ce'); log_m = np.log(m.full_mean)
    f = pd.DataFrame(index=m.index)
    for w in (1,2,3): f[f'{prefix}_log_change_{w}m_pct'] = 100 * (log_m - log_m.shift(w))
    f[f'{prefix}_volatility_6m_pct'] = (100 * log_m.diff()).rolling(6, min_periods=6).std(ddof=1)
    f[f'{prefix}_lead_t1_full_pct'] = 100 * (np.log(m.full_mean.shift(-1)) - log_m)
    f[f'{prefix}_lead_t1_cutoff24_pct'] = 100 * (np.log(m.cutoff24_mean.shift(-1)) - log_m)
    f[f'{prefix}_lead_t1_full_max_date'] = m.full_max_date.shift(-1)
    f[f'{prefix}_lead_t1_cutoff24_max_date'] = m.cutoff24_max_date.shift(-1)
    base = base.merge(f.reset_index(), on='month_ce', how='left', validate='many_to_one')

rows = []
for h in HORIZONS:
    p = base.copy(); p['horizon'] = h
    p['forecast_origin'] = p.month_ce + pd.offsets.MonthBegin(2) + pd.Timedelta(days=14)
    p['target_month'] = p.month_ce + pd.offsets.MonthBegin(h)
    p['target_imi_index'] = p.groupby('commodity_code').imi_index.shift(-h)
    p['target_y_pct'] = 100 * (np.log(p.target_imi_index) - np.log(p.imi_index))
    p['target_available_date'] = p.target_month + pd.offsets.MonthBegin(2) + pd.Timedelta(days=14)
    rows.append(p)
IMI = [f'imi_log_change_{w}m_pct' for w in (1,2,3,6,12)] + ['imi_volatility_6m_pct']
FX = [f'fx_log_change_{w}m_pct' for w in (1,2,3)] + ['fx_volatility_6m_pct']
BRENT = [f'brent_log_change_{w}m_pct' for w in (1,2,3)] + ['brent_volatility_6m_pct']
MODEL_FEATURES = IMI + FX + BRENT + ['fx_lead_t1_full_pct','brent_lead_t1_full_pct',
                                     'fx_lead_t1_cutoff24_pct','brent_lead_t1_cutoff24_pct']
modeling = pd.concat(rows, ignore_index=True).dropna(subset=MODEL_FEATURES + ['target_y_pct'])
modeling = modeling.sort_values(['horizon','commodity_code','month_ce']).reset_index(drop=True)
first_t, last_imi = pd.Timestamp('2001-01-01'), imi.month_ce.max()
expected_rows = {h: len(pd.date_range(first_t, last_imi - pd.DateOffset(months=h), freq='MS')) for h in HORIZONS}
assert len(modeling) == len(GROUPS) * sum(expected_rows.values()), (len(modeling), expected_rows)
assert modeling.month_ce.min() == first_t
FEATURE_SETS = {'1': IMI, '2': IMI + FX + ['fx_lead_t1_full_pct'],
 '3': IMI + FX + BRENT + ['fx_lead_t1_full_pct','brent_lead_t1_full_pct'],
 '3-noRecent': IMI + FX + BRENT,
 '3-cutoff24': IMI + FX + BRENT + ['fx_lead_t1_cutoff24_pct','brent_lead_t1_cutoff24_pct']}
feature_sets = pd.DataFrame([{'feature_set':k,'n_features':len(v),'features':'|'.join(v)} for k,v in FEATURE_SETS.items()])

# 4) Leakage audit
expected_origin = modeling.month_ce + pd.offsets.MonthBegin(2) + pd.Timedelta(days=14)
expected_target = pd.Series([m + pd.offsets.MonthBegin(int(h)) for m,h in zip(modeling.month_ce, modeling.horizon)], index=modeling.index)
masks = {'forecast_origin_formula': modeling.forecast_origin != expected_origin,
 'target_month_formula': modeling.target_month != expected_target,
 'target_already_available_at_origin': modeling.target_available_date <= modeling.forecast_origin,
 'fx_full_observation_not_before_origin': pd.to_datetime(modeling.fx_lead_t1_full_max_date) >= modeling.forecast_origin,
 'brent_full_observation_not_before_origin': pd.to_datetime(modeling.brent_lead_t1_full_max_date) >= modeling.forecast_origin,
 'fx_cutoff24_not_before_origin': pd.to_datetime(modeling.fx_lead_t1_cutoff24_max_date) >= modeling.forecast_origin,
 'brent_cutoff24_not_before_origin': pd.to_datetime(modeling.brent_lead_t1_cutoff24_max_date) >= modeling.forecast_origin,
 'required_feature_missing': modeling[MODEL_FEATURES].isna().any(axis=1)}
audit = pd.DataFrame([{'check':k,'violations':int(v.sum()),'passed':int(v.sum()) == 0} for k,v in masks.items()])
assert audit.passed.all(), audit
test_counts = modeling[modeling.month_ce >= '2019-01-01'].groupby(['horizon','commodity_code']).size().rename('test_origins').reset_index()
expected_test = {h: len(pd.date_range('2019-01-01', last_imi - pd.DateOffset(months=h), freq='MS')) for h in HORIZONS}
for h in HORIZONS: assert set(test_counts.query('horizon == @h').test_origins) == {expected_test[h]}
assert expected_test[3] == 89 and expected_test[2] == 90
display(feature_sets); display(audit); display(test_counts)

# 5) บันทึก derived data, metrics และ process log
out = PROCESSED_DIR / 'modeling_table.csv'; modeling.to_csv(out, index=False)
# feature ของเดือนฐานล่าสุดที่ยังไม่มี target ใช้สำหรับพยากรณ์ล่วงหน้าจริงในขั้น 06
latest = base[base[MODEL_FEATURES].notna().all(axis=1) & (base.month_ce > modeling.month_ce.max() - pd.DateOffset(months=1))].copy()
latest['forecast_origin'] = latest.month_ce + pd.offsets.MonthBegin(2) + pd.Timedelta(days=14)
latest.to_csv(PROCESSED_DIR / 'latest_features.csv', index=False)
counts = modeling.groupby(['horizon','commodity_code']).agg(rows=('target_y_pct','size'), first_t=('month_ce','min'), last_t=('month_ce','max')).reset_index().merge(test_counts)
counts.to_csv(METRICS_DIR / '03_processed_row_counts.csv', index=False)
feature_sets.to_csv(METRICS_DIR / '03_feature_sets.csv', index=False)
audit.to_csv(METRICS_DIR / '03_leakage_audit.csv', index=False)
modeling[['commodity_code','month_ce','horizon','forecast_origin','target_month','target_available_date',
          'fx_lead_t1_full_max_date','brent_lead_t1_full_max_date']].tail(12).to_csv(METRICS_DIR / '03_leakage_audit_sample.csv', index=False)
manifest = pd.DataFrame([{'file':str(out.relative_to(ROOT)),'rows':len(modeling),'sha256':sha256(out),
 'forecast_origin_rule':'day 15 of t+2','primary_horizon':3,'diagnostic_horizon':2,'rq4_horizons':'4|5|6','release_calendar_records':len(release)}])
manifest.to_csv(METRICS_DIR / '03_processed_manifest.csv', index=False)
log_path = ROOT / 'logs/process_log.md'; log = log_path.read_text(encoding='utf-8')
marker = 'ขั้นตอน: 03 Data Preparation (แก้ไข availability)'
if marker not in log:
    entry = f'''\n\n## [2026-10-05] ขั้นตอน: 03 Data Preparation (แก้ไข availability)
**สิ่งที่ทำ:** ตรวจ archive ข่าวเผยแพร่ IMI ของ สนค. และสร้าง feature/target ใหม่ โดยกำหนด forecast origin วันที่ 15 ของ t+2 ใช้ h=3 เป็น target หลักและ h=2 เป็น diagnostic ค่าเฉลี่ยข้อมูลภายนอกเต็มเดือน t+1 เป็น recent-information feature และ cutoff วันที่ 24 เป็น sensitivity

**ผลที่ได้:** archive ที่ตรวจได้มี {len(release)} เดือน release lag {release.lag_days_after_month_end.min()}–{release.lag_days_after_month_end.max()} วัน (มัธยฐาน {release.lag_days_after_month_end.median():.0f} วัน) ตาราง processed ใหม่มี {len(modeling):,} แถว; h=3 มี 305 แถวต่อกลุ่มและ 89 test origins ส่วน h=2 มี 306 แถวต่อกลุ่มและ 90 test origins ไม่พบ violation ใน leakage audit

**สิ่งที่พบ / ปัญหา:** สมมติฐานเดิมว่าวันที่ 25 ของ t+1 มี IMI เดือน t และข้อมูลภายนอกถึงวันที่ 24 พร้อมใช้ไม่เป็นจริงอย่างสม่ำเสมอ ผลจาก Data Preparation รุ่นเดิมจึงยกเลิกใช้ ปฏิทินเผยแพร่ IMI ที่ยืนยันได้ครอบคลุมเพียง พ.ย. 2567–ก.ค. 2569 และ FRED ไม่มี historical vintage availability ราย observation ในไฟล์ที่ดาวน์โหลด

**การตัดสินใจและเหตุผล:** ใช้วันที่ 15 ของ t+2 เป็น cutoff แบบอนุรักษนิยม เพราะทั้ง 21 เดือนที่ตรวจเผยแพร่ไม่เกินวันที่ 10 ของ t+2 จึงมี buffer อย่างน้อย 5 วัน ขยับผลหลักเป็น h=3 เพื่อคงประโยชน์การเตือนเดือนถัดไป และใช้ target availability rule เดียวกันในการคัด training rows ของ rolling origin

**ขั้นต่อไป:** ดำเนินการ 04 Modeling ด้วยตารางที่แก้ไขแล้ว และบังคับให้แต่ละ origin ใช้เฉพาะ training rows ที่ target_available_date ไม่เกิน forecast origin ของรอบนั้น
'''
    log_path.write_text(log.rstrip() + entry + '\n', encoding='utf-8')
    log = log_path.read_text(encoding='utf-8')
marker_h = 'ขั้นตอน: 03 Data Preparation (เพิ่ม horizon h=4-6 สำหรับ RQ4)'
if marker_h not in log:
    rows_txt = ', '.join(f'h={h} {expected_rows[h]} แถว/{expected_test[h]} test' for h in HORIZONS)
    entry_h = f"""

## [2026-10-05] {marker_h}
**สิ่งที่ทำ:** ขยาย horizon จาก h=2-3 เป็น h=2-6 ตาม Scope RQ4(ก) โดยไม่เปลี่ยน feature, forecast origin หรือกติกา target availability ทำก่อนเริ่มขั้น Modeling

**ผลที่ได้:** ตาราง processed มี {len(modeling):,} แถว (ต่อกลุ่ม: {rows_txt}) leakage audit ผ่านทั้ง {len(audit)} เงื่อนไข SHA-256 {sha256(out)[:12]}...

**สิ่งที่พบ / ปัญหา:** แถวเพิ่มเฉพาะ horizon ใหม่ แถวของ h=2 และ h=3 เท่าเดิม (306 และ 305 แถวต่อกลุ่ม)

**การตัดสินใจและเหตุผล:** h=3 ยังเป็นผลหลักและใช้ในการเปรียบเทียบหลัก 3 ข้อ ส่วน h=4-6 ใช้วาดเส้นความแม่นตาม horizon เป็นผลเชิงสำรวจ เพื่อตอบว่าระบบเตือนได้ไกลแค่ไหน

**ขั้นต่อไป:** 04 Modeling ตามขั้นตอน rolling origin ใน Scope หัวข้อ 12
"""
    log_path.write_text(log.rstrip() + entry_h + '\n', encoding='utf-8')
display(manifest); print(f'บันทึก {out.relative_to(ROOT)} และ metrics ขั้น 03 เรียบร้อย')

,source,file,sha256_ok
0,IMI,imi_monthly_20261005T054207Z.csv,True
1,DEXTHUS,dexthus_daily_20261005T054207Z.csv,True
2,DCOILBRENTEU,dcoilbrenteu_daily_20261005T054207Z.csv,True
3,TPSO_RELEASE,tpso_imi_release_calendar_20261005T065618Z.csv,True


,records,reference_start,reference_end,lag_min_days,lag_median_days,lag_max_days,latest_release_day_when_in_t2,minimum_buffer_to_day15
0,21,2024-11-01,2026-07-01,26,29.0,41,10,5


,months,start,end,min_full_n,min_cutoff24_n
series,,,,,
DCOILBRENTEU,321,2000-01-01,2026-09-01,9,7
DEXTHUS,321,2000-01-01,2026-09-01,18,13


,feature_set,n_features,features
0,1,6,imi_log_change_1m_pct|imi_log_change_2m_pct|im...
1,2,11,imi_log_change_1m_pct|imi_log_change_2m_pct|im...
2,3,16,imi_log_change_1m_pct|imi_log_change_2m_pct|im...
3,3-noRecent,14,imi_log_change_1m_pct|imi_log_change_2m_pct|im...
4,3-cutoff24,16,imi_log_change_1m_pct|imi_log_change_2m_pct|im...


,check,violations,passed
0,forecast_origin_formula,0,True
1,target_month_formula,0,True
2,target_already_available_at_origin,0,True
3,fx_full_observation_not_before_origin,0,True
4,brent_full_observation_not_before_origin,0,True
5,fx_cutoff24_not_before_origin,0,True
6,brent_cutoff24_not_before_origin,0,True
7,required_feature_missing,0,True


,horizon,commodity_code,test_origins
0,2,100,90
1,2,318,90
2,2,402,90
3,3,100,89
4,3,318,89
5,3,402,89
6,4,100,88
7,4,318,88
8,4,402,88
9,5,100,87


,file,rows,sha256,forecast_origin_rule,primary_horizon,diagnostic_horizon,rq4_horizons,release_calendar_records
0,data/processed/modeling_table.csv,4560,c97d79ab0c1afade35d0f1bcf999d51b4c0b58c774048c...,day 15 of t+2,3,2,4|5|6,21


บันทึก data/processed/modeling_table.csv และ metrics ขั้น 03 เรียบร้อย
